In [1]:
import sys
import os
import pandas as pd
import glob
import shutil
import time
import job_manager

import numpy as np
import subprocess

import ase.atoms
import ase.visualize

import arkane.encorr.reference
import arkane.encorr.corr
import arkane.ess
import arkane.encorr.bac
import arkane.exceptions
import arkane.common
import rmgpy.molecule

sys.path.append(os.environ['DFT_DIR'])
import autotst_wrapper

sys.path.append(os.environ['DATABASE_DIR'])
import database_fun


import collections

# log to ethalpy
# from collections import defaultdict, Counter
# import os
# import re

import rdkit.Chem # import GetPeriodicTable

import rmgpy.quantity # import ScalarQuantity
import rmgpy.statmech #import HarmonicOscillator, IdealGasTranslation, LinearRotor, NonlinearRotor
import rmgpy.thermo #import ThermoData

# from arkane.common import symbol_by_number
# from arkane.encorr.corr import get_atom_correction, assign_frequency_scale_factor
# from arkane.encorr.reference import CalculatedDataEntry, ReferenceDatabase
import arkane.modelchem  # import LevelOfTheory, CompositeLevelOfTheory


hotbit not installed
Loading DFT database from /projects/westgroup/harris.se/autoscience/reaction_calculator/database


In [ ]:
# load the reference database
database = arkane.encorr.reference.ReferenceDatabase()
database.load()

*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(2)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(2)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): O(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): O(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): O+1(2)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): O+1(2)
ERROR:root:Unable to generate identifier for this molecule:
1 O u0 p3 c-1 {3,S}
2 O u0 p2 c0 {3,D}
3 C u0 p0 c0 {1,S} {2,D} {4,S}
4 H u0 p0 c0 {3,S}

*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valen

*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): S(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): S(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): S(1); O(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): S(1); O(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): O(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): O(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): O(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): O(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted

In [ ]:
database.reference_sets['main'][1]

In [ ]:
query_sp = rmgpy.species.Species(smiles='C[CH]CC')
for i in range(len(database.reference_sets['main'])):
    ref_sp = rmgpy.species.Species(smiles=database.reference_sets['main'][i].smiles)
    if ref_sp.is_isomorphic(query_sp):
        print(i)
        break

In [ ]:
query_sp = rmgpy.species.Species(smiles='[O][O]')
for i in range(len(database.reference_sets['main'])):
    ref_sp = rmgpy.species.Species(smiles=database.reference_sets['main'][i].smiles)
    if ref_sp.is_isomorphic(query_sp):
        print(i)
        break

In [ ]:
# what's in the database that seems the most like my secondary butane radical?

In [ ]:
ethyne = rmgpy.species.Species(smiles='C#C')

In [ ]:
benzyl = rmgpy.species.Species(smiles='[CH2]C1=CC=CC=C1')

In [ ]:
toluene = rmgpy.species.Species(smiles='CC1=CC=CC=C1')

In [ ]:
for i in range(len(database.reference_sets['main'])):
    ref_sp = rmgpy.species.Species(smiles=database.reference_sets['main'][i].smiles)
    if ref_sp.is_isomorphic(ethyne):
        print(i)
        break

# Only collect uncharged C,H,O species

In [ ]:
# Redo Single Points where geometries don't match corresponding gaussian files
special_set = []
for i in range(len(database.reference_sets['main'])):
    if database.reference_sets['main'][i].charge != 0:
        continue
    if 'N' in database.reference_sets['main'][i].smiles.upper():
        continue
    if 'S' in database.reference_sets['main'][i].smiles.upper():
        continue
    if 'CL' in database.reference_sets['main'][i].smiles.upper():
        continue
    if 'BR' in database.reference_sets['main'][i].smiles.upper():
        continue
    if 'F' in database.reference_sets['main'][i].smiles.upper():
        continue
    special_set.append(i)

In [ ]:
working_dir = '/scratch/harris.se/guassian_scratch/bac_20250328'

# Check progress on geometry optimizations

In [ ]:
def has_right_modes(cf):
    if not cf.modes:
        return False
    elif not any(isinstance(mode, rmgpy.statmech.IdealGasTranslation) for mode in cf.modes):
        return False
    elif not any(isinstance(mode, (rmgpy.statmech.LinearRotor, rmgpy.statmech.NonlinearRotor)) for mode in cf.modes):
        return False
    elif not any(isinstance(mode, rmgpy.statmech.HarmonicOscillator) for mode in cf.modes):
        return False
    return True

In [ ]:
working_dir = '/scratch/harris.se/guassian_scratch/bac_20250328'

incomplete_geo = []
bad_geo = []
complete_geo = []

# for i in range(len(database.reference_sets['main'])):
for i in special_set:
    sp_logfile = os.path.join(working_dir, f'species_{i:04}', 'freq.log')
    if not os.path.exists(sp_logfile):
        incomplete_geo.append(i)
        continue
    try:
        gl = arkane.ess.factory.ess_factory(sp_logfile)
    except arkane.exceptions.LogError:
        incomplete_geo.append(i)
        continue
    
    cf, freqs = gl.load_conformer()
    if not has_right_modes(cf):
        bad_geo.append(i)
    elif gl.load_force_constant_matrix() is None:
        bad_geo.append(i)
    else:
        complete_geo.append(i)

In [ ]:
print(len(complete_geo))

In [ ]:
print(len(bad_geo))

In [ ]:
# bad_geo

In [ ]:
print(len(incomplete_geo))

In [ ]:
incomplete_geo

### Add check for consistent hessian ###

In [ ]:
inconsistent = []
for i in complete_geo:
    ref_sp = rmgpy.species.Species(smiles=database.reference_sets['main'][i].smiles)
    if len(ref_sp.molecule[0].atoms) < 14:
        continue  # the check doesn't really work on these
    
    if i in [157]:  # 157 has no rotors (cylo-pentane), so this check doesn't mean anything
        continue
    
    sp_logfile = os.path.join(working_dir, f'species_{i:04}', 'freq.log')
    print(i)
    try:
        if not autotst_wrapper.check_hessian_cartesian_consistent(sp_logfile):
            inconsistent.append(i)
            print(i, '\tinconsistent')
    except ValueError:
        inconsistent.append(i)
        print(i, '\tFailed')
        

# Make geometry optimization files from existing reference database geometries

In [ ]:
for i in range(len(ref_db.reference_sets['main'])):

    # get starting geometry from previous calculations
    preferred_methods = ['ccsd(t)f12', 'cbsqb32023']
    atoms = None
    for preferred_method in preferred_methods:
        for key in database.reference_sets['main'][i].calculated_data.keys():
            comparison = key
            if type(key) != arkane.modelchem.LevelOfTheory:
                comparison = key.energy
            
            if comparison.method == preferred_method:
                syms = database.reference_sets['main'][i].calculated_data[key].xyz_dict['symbols']
                xyz = database.reference_sets['main'][i].calculated_data[key].xyz_dict['coords']
                atoms = ase.Atoms(symbols=syms, positions=xyz)
                break
        if atoms:
            break
    else:
        raise ValueError(f'no preferred level of theory for entry {i}')


    sp_dir = os.path.join(working_dir, f'species_{i:04}')
    os.makedirs(sp_dir, exist_ok=True)
    
    # write the gaussian calculation file
    with open(os.path.join(sp_dir, 'sp.com'), 'w') as f:
        ase.io.gaussian.write_gaussian_in(
            f,
            atoms,
            properties=['energy'],
            method='m062x',
            basis='cc-pvtz',
            mult=database.reference_sets['main'][i].multiplicity,
            charge=database.reference_sets['main'][i].charge,
            extra ="opt=(calcfc,maxcycles=900,tight) IOP(7/33=1,2/9=2000,2/16=3) scf=(maxcycle=900,tight,direct) integral=(grid=ultrafine, Acc2E=12)",
        )

# write the slurm script
run_script = os.path.join(working_dir, 'run.sh')
with open(run_script, 'w') as f:
    f.write("""#!/bin/bash
#SBATCH --job-name=g16_bac_opt
#SBATCH --error=error.log
#SBATCH --nodes=1
#SBATCH --partition=west,short
#SBATCH --exclude=c5003
#SBATCH --mem=20Gb
#SBATCH --time=24:00:00
#SBATCH --cpus-per-task=16
""" +
f'#SBATCH --array={autotst_wrapper.ordered_array_str(my_indices)}%10\n' +
"""
export GAUSS_SCRDIR=/scratch/harris.se/guassian_scratch
mkdir -p $GAUSS_SCRDIR
module load gaussian/g16
source /shared/centos7/gaussian/g16/bsd/g16.profile

RUN_i=$(printf "%04.0f" $(($SLURM_ARRAY_TASK_ID)))

cd "species_${RUN_i}"
g16 sp.com

""")


In [ ]:
os.path.exists(prev_sp_calc)

# Make geometry optimization files from previous bac attempt

In [ ]:
my_indices = []
prev_working_dir = '/scratch/harris.se/guassian_scratch/bac_20250326'
# for i in special_set:
for i in range(len(database.reference_sets['main'])):
    prev_sp_calc = os.path.join(prev_working_dir, f'species_{i:04}', 'sp.log')
    if not os.path.exists(prev_sp_calc):
        continue
    try:
        gl = arkane.ess.factory.ess_factory(prev_sp_calc)
        coords, num, mass = arkane.ess.factory.ess_factory(prev_sp_calc).load_geometry()
        atoms = ase.Atoms(num, coords)
    except (arkane.exceptions.LogError, FileNotFoundError):
        continue

    sp_dir = os.path.join(working_dir, f'species_{i:04}')
    os.makedirs(sp_dir, exist_ok=True)
    
    # write the gaussian calculation file
    with open(os.path.join(sp_dir, 'sp.com'), 'w') as f:
        ase.io.gaussian.write_gaussian_in(
            f,
            atoms,
            properties=['energy'],
            method='m062x',
            basis='cc-pvtz',
            mult=database.reference_sets['main'][i].multiplicity,
            charge=database.reference_sets['main'][i].charge,
            extra ="opt=(calcfc,maxcycles=900,tight) IOP(7/33=1,2/9=2000,2/16=3) scf=(maxcycle=900,tight,direct) integral=(grid=ultrafine, Acc2E=12)",
        )
    my_indices.append(i)

# write the slurm script
run_script = os.path.join(working_dir, 'run.sh')
with open(run_script, 'w') as f:
    f.write("""#!/bin/bash
#SBATCH --job-name=g16_bac_opt
#SBATCH --error=error.log
#SBATCH --nodes=1
#SBATCH --partition=west,short
#SBATCH --exclude=c5003
#SBATCH --mem=20Gb
#SBATCH --time=24:00:00
#SBATCH --cpus-per-task=16
""" +
f'#SBATCH --array={autotst_wrapper.ordered_array_str(my_indices)}%10\n' +
"""
export GAUSS_SCRDIR=/scratch/harris.se/guassian_scratch
mkdir -p $GAUSS_SCRDIR
module load gaussian/g16
source /shared/centos7/gaussian/g16/bsd/g16.profile

RUN_i=$(printf "%04.0f" $(($SLURM_ARRAY_TASK_ID)))

cd "species_${RUN_i}"
g16 sp.com

""")


In [ ]:
my_indices

### See what's missing from the autoscience database

In [ ]:
not_in_db = []
for i in bad_geo:
    ref_sp = rmgpy.species.Species().from_adjacency_list(database.reference_sets['main'][i].adjacency_list)
    try:
        db_index = database_fun.get_unique_species_index(ref_sp)
    except IndexError:
        print(i, ref_sp.smiles, 'not in db')
        not_in_db.append(ref_sp)
        continue

In [ ]:
not_in_db = [rmgpy.species.Species(smiles='C#CC#C')]

In [ ]:
# database_fun.add_species_to_database(not_in_db)

In [ ]:
bad_geo

# Copy completed geometry optimizations from autoscience database

In [ ]:
# for i in bad_geo:
skiplist = []
for i in bad_geo:
    ref_sp = rmgpy.species.Species().from_adjacency_list(database.reference_sets['main'][i].adjacency_list)
    try:
        db_index = database_fun.get_unique_species_index(ref_sp)
    except IndexError:
        print(i, ref_sp.smiles, 'not in db')
        not_in_db.append(ref_sp)
        continue
        
    if db_index in skiplist:
        print('skipping', i)
        continue
        
    print(f'Looking for db index {db_index}')
    # see if there's a complete file
    try:
        rotor_logfile = glob.glob(os.path.join(os.environ['DFT_DIR'], 'thermo', f'species_{db_index:04}', 'rotors', 'conformer_*.log'))[0]
    except IndexError:
        rotor_logfile = None
        
    try:
        iop_logfile = glob.glob(os.path.join(os.environ['DFT_DIR'], 'thermo', f'species_{db_index:04}', 'rotors', 'iop_recalc_*.log'))[0]
    except IndexError:
        iop_logfile = None
        
    try:
        arkane_logfile = glob.glob(os.path.join(os.environ['DFT_DIR'], 'thermo', f'species_{db_index:04}', 'arkane', 'conformer_*.log'))[0]
    except IndexError:
        arkane_logfile = None
    
    my_logfile = None
    if iop_logfile is not None:
        try:
            gl = arkane.ess.factory.ess_factory(iop_logfile)
            cf, freqs = gl.load_conformer()
            if not has_right_modes(cf):
                raise arkane.exceptions.LogError
            elif gl.load_force_constant_matrix() is None:
                arkane.exceptions.LogError
            my_logfile = iop_logfile
        except arkane.exceptions.LogError:
            print(f'Bad IOP')
            continue
    elif rotor_logfile is not None:
        try:
            gl = arkane.ess.factory.ess_factory(rotor_logfile)
            cf, freqs = gl.load_conformer()
            if not has_right_modes(cf):
                raise arkane.exceptions.LogError
            elif gl.load_force_constant_matrix() is None:
                arkane.exceptions.LogError
            my_logfile = rotor_logfile
        except arkane.exceptions.LogError:
            print(f'Bad Rotor')
            continue
    elif arkane_logfile is not None:
        try:
            gl = arkane.ess.factory.ess_factory(arkane_logfile)
            cf, freqs = gl.load_conformer()
            if not has_right_modes(cf):
                raise arkane.exceptions.LogError
            elif gl.load_force_constant_matrix() is None:
                arkane.exceptions.LogError
            my_logfile = arkane_logfile
        except arkane.exceptions.LogError:
            print(f'Bad Arkane')
            continue
    else:
        print('nothing to copy')
        continue
    
    dest_file = os.path.join(working_dir, f'species_{i:04}', 'sp.log')
    print(f'copying {my_logfile} to {dest_file}')
    shutil.copyfile(my_logfile, dest_file)
    

In [ ]:
species_index = 1027
conformer_dir = os.path.join(os.environ['DFT_DIR'], 'thermo', f'species_{species_index:04}', 'conformers')
rotor_dir = os.path.join(os.environ['DFT_DIR'], 'thermo', f'species_{species_index:04}', 'rotors')
conformer_file = autotst_wrapper.get_lowest_valid_conformer(conformer_dir)

# conformer_file = autotst_wrapper.get_lowest_energy_gaussian_file(conformer_dir)

os.makedirs(rotor_dir, exist_ok=True)
shutil.copyfile(conformer_file, os.path.join(rotor_dir, os.path.basename(conformer_file)))

In [ ]:
inconsistent

In [ ]:
bad_geo

In [ ]:
incomplete_geo

# Check progress on single-point calculations

In [ ]:
incomplete_sp = []
bad_sp = []
complete_sp = []

# for i in range(len(database.reference_sets['main'])):
for i in special_set:
#     if i in inconsistent or i in bad_geo or i in incomplete_geo:
#         continue
    
    
    orca_logfile = os.path.join(working_dir, f'species_{i:04}', 'conformer.out')
    if not os.path.exists(orca_logfile):
        incomplete_sp.append(i)
        continue
    try:
        ol = arkane.ess.factory.ess_factory(orca_logfile)
        ol.load_energy()
    except arkane.exceptions.LogError:
        incomplete_sp.append(i)
        continue
        
    # make sure that the coordinates match between orca and Gaussian
    sp_logfile = os.path.join(working_dir, f'species_{i:04}', 'sp.log')
    
    g_coords, g_num, g_mass = arkane.ess.factory.ess_factory(sp_logfile).load_geometry()
    o_coords, o_num, o_mass = ol.load_geometry()
    
    if not np.all(np.equal(np.array(g_coords), np.array(o_coords))):
        g2_coords, g_num, g_mass = arkane.ess.factory.ess_factory(sp_logfile).load_geometry()
        if not np.all(np.equal(np.array(g2_coords), np.array(o_coords))):
            bad_sp.append(i)
            print(f'problem with {i}')
        else:
            complete_sp.append(i)
    else:
        complete_sp.append(i)
    

In [ ]:
len(complete_sp)

In [ ]:
len(incomplete_sp)

In [ ]:
incomplete_sp

In [ ]:
len(complete_sp)

In [ ]:
len(bad_sp)

In [ ]:
bad_sp

In [ ]:
working_dir

# Delete old single point calculation files

In [ ]:
# for i in bad_sp:
for i in [115]:
    cf_files = glob.glob(os.path.join(working_dir, f'species_{i:04}', 'conformer.*'))
    for j in cf_files:
        os.remove(j)

# Remake single point files

In [ ]:
special_set[0]

In [ ]:
# for i in bad_sp + incomplete_sp:
# for i in bad_sp:
# for i in [115]:
# for i in special_set:
# for i in range(len(database.reference_sets['main'])):
for i in [101]:

    sp_dir = os.path.join(working_dir, f'species_{i:04}')
    gaussian_log = os.path.join(sp_dir, 'sp.log')
    # make an orca file
    
    try:
        my_log = arkane.ess.ess_factory(gaussian_log)
    except arkane.exceptions.LogError:
        print(f'skipping bad gaussian file {i}')
        continue
    # make a run file
    coord, number, mass = my_log.load_geometry()
    my_atoms = ase.Atoms(number, coord)
    
    
    parallel = True
    orca_input_file = os.path.join(sp_dir, 'conformer.inp')
    input_format = """!{res}HF {level_of_theory} TightSCF tightPNO
!energy

%maxcore 35000
{opt_parallel_line}

* xyz {charge} {mult}
{xyz}*


"""
    charge = database.reference_sets['main'][i].charge
    multiplicity = database.reference_sets['main'][i].multiplicity
    input_content = input_format.format(
        res='r' if multiplicity == 1 else 'u',
        level_of_theory='dlpno-ccsd(t)-f12 cc-pvtz-f12 aug-cc-pvtz/c cc-pvtz-f12-cabs',
        opt_parallel_line='%pal nprocs 4 end' if parallel else '',
        charge=charge,
        mult=multiplicity,
        xyz=autotst_wrapper.get_xyz(my_atoms),
    )
    
    with open(orca_input_file, 'w') as f:
        f.write(input_content)
        
    run_orca_script = os.path.join(sp_dir, 'run_orca.sh')
    with open(run_orca_script, 'w') as f:
        # TODO format the text without """ so it doesn't mess with VSCode's collapse function button
        f.write("""#!/bin/bash
#SBATCH --job-name=""" + f'orca_{i:04}' + """
#SBATCH --error=error.log
#SBATCH --nodes=1
#SBATCH --partition=west,short
#SBATCH --exclude=c5003
#SBATCH --mem-per-cpu=50Gb
#SBATCH --time=24:00:00
#SBATCH --ntasks=4


ompi=/work/westgroup/orca/openmpi-4.1.6/build
PATH=$ompi/bin:$PATH
LD_LIBRARY_PATH=$ompi/lib:$ompi/etc:$LD_LIBRARY_PATH

#Orca
orcadir=/work/westgroup/orca/orca_6_0_1_linux_x86-64_shared_openmpi416
export PATH=$PATH:$orcadir
export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:$orcadir

$orcadir/orca conformer.inp > conformer.out
""")

In [ ]:
my_calcs = sorted(bad_sp + incomplete_sp)

In [ ]:
autotst_wrapper.ordered_array_str(my_calcs)

In [ ]:
print(my_calcs)

# Make Frequency Runfiles

In [ ]:
incomplete_sp = []

# for i in special_set:
# for i in range(len(database.reference_sets['main'])):
# for i in to_calculate:
# for i in to_calculate:
for i in [101]:
#     if i in inconsistent or i in bad_geo or i in incomplete_geo:
#         continue
    conformer_file = os.path.join(working_dir, f'species_{i:04}', 'sp.log')
    freq_comfile = os.path.join(working_dir, f'species_{i:04}', 'freq.com')
    try:
        gaussian_logfile = arkane.ess.gaussian.GaussianLog(conformer_file)
        coord, number, mass = gaussian_logfile.load_geometry()
    except arkane.exceptions.LogError:
        incomplete_sp.append(i)
        print(f'Bad geometry file {i}')
        continue
    atoms = ase.Atoms(positions=coord, symbols=number)

    rmg_species = rmgpy.species.Species().from_adjacency_list(database.reference_sets['main'][i].adjacency_list)
    
    # write the gaussian calculation file
    with open(freq_comfile, 'w') as f:
        ase.io.gaussian.write_gaussian_in(
            f,
            atoms,
            properties=['energy'],
            method='m062x',
            basis='cc-pvtz',
            mult=rmg_species.multiplicity,
            charge=rmg_species.get_net_charge(),
            extra=f'freq IOP(7/33=1,2/9=2000) scf=(tight,direct)',
            mem='15GB',
            nprocshared=24,
        )

    # write the slurm script
    run_script = os.path.join(working_dir, f'species_{i:04}', 'run_freq.sh')
    with open(run_script, 'w') as f:
        f.write("""#!/bin/bash
#SBATCH --job-name=freq_""" + str(i) + """
#SBATCH --error=error.log
#SBATCH --nodes=1
#SBATCH --partition=west,short
#SBATCH --exclude=c5003
#SBATCH --mem=20Gb
#SBATCH --time=24:00:00
#SBATCH --ntasks=24

export GAUSS_SCRDIR=/scratch/harris.se/guassian_scratch
mkdir -p $GAUSS_SCRDIR
module load gaussian/g16
source /shared/centos7/gaussian/g16/bsd/g16.profile


g16 freq.com

""")


# Run Frequency Runfiles

In [ ]:
MAX_JOBS_RUNNING = 20
# for i in [418]:
# for i in to_calculate[0:4]:
for i in range(len(database.reference_sets['main'])):
    start_dir = os.getcwd()
    freq_dir = os.path.join(working_dir, f'species_{i:04}')
    
    freq_logfile = os.path.join(working_dir, f'species_{i:04}', 'freq.log')
    try:
        gaussian_logfile = arkane.ess.gaussian.GaussianLog(freq_logfile)
        coord, number, mass = gaussian_logfile.load_geometry()
        cf, freqs = gaussian_logfile.load_conformer()
        print(f'{i} already computed')
        continue
    except (arkane.exceptions.LogError, FileNotFoundError):
        pass
    
    
    run_freq_script = os.path.join(freq_dir, 'run_freq.sh')
    os.chdir(freq_dir)
    freq_job = job_manager.SlurmJob()
    slurm_cmd = f"sbatch {run_freq_script}"

    # wait for fewer than MAX_JOBS_RUNNING jobs running
    jobs_running = job_manager.count_slurm_jobs()
    while jobs_running > MAX_JOBS_RUNNING:
        time.sleep(60)
        jobs_running = job_manager.count_slurm_jobs()

    freq_job.submit(slurm_cmd)
    time.sleep(5.0)
    os.chdir(start_dir)
    
    

## Check completed vs yet to run

In [ ]:
completed_freq = []

for i in special_set:
    freq_logfile = os.path.join(working_dir, f'species_{i:04}', 'freq.log')
    try:
        gaussian_logfile = arkane.ess.gaussian.GaussianLog(freq_logfile)
        coord, number, mass = gaussian_logfile.load_geometry()
#         cf, freqs = gaussian_logfile.load_conformer()
#         print(f'{i} already computed')
        completed_freq.append(i)
        continue
    except (arkane.exceptions.LogError, FileNotFoundError):
        pass

In [ ]:
to_calculate = sorted(list(set(special_set) - set(completed_freq)))

In [ ]:
to_calculate

In [ ]:
len(to_calculate)

In [ ]:
[42, 101, 153, 157, 384, 408, 411, 413, 415, 416, 419]

In [ ]:
total_incomplete = sorted(list(set(special_set) - set(complete_sp)))

In [ ]:
total_incomplete

In [ ]:
for i in total_incomplete:
    ref_sp = rmgpy.species.Species().from_adjacency_list(database.reference_sets['main'][i].adjacency_list)
    print(i, '\t', database_fun.get_unique_species_index(ref_sp), database.reference_sets['main'][i].smiles)

In [ ]:
inconsistent

In [ ]:
bad_geo

In [ ]:
bad_sp

In [ ]:
database.reference_sets['main'][100]

## Define things for addition to database

In [ ]:
method = 'M062X/ccpvtz'
freq_lot = arkane.modelchem.LevelOfTheory(method='M062X2023',
                           basis='ccpvtz',
                           software='gaussian',
                           )
energy_lot = arkane.modelchem.LevelOfTheory(method='dlpno-ccsd(t)-f12-2023',
                           basis='ccpvtzf12',
                           software='orca',
                           )

periodic_table = rdkit.Chem.GetPeriodicTable()

def log_to_xyz_dict(log):
    log = arkane.ess.ess_factory(log)
    coords, nums, _ = log.load_geometry()
    syms = [arkane.common.symbol_by_number[int(n)] for n in nums]
    return {
        'coords': coords,
        'isotopes': [periodic_table.GetMostCommonIsotope(s) for s in syms],
        'symbols': syms
    }

level_of_theories = {
    'dlpno-ccsdt-f12-ccpvtz': arkane.modelchem.CompositeLevelOfTheory(freq=arkane.modelchem.LevelOfTheory(method='m062x', basis='ccpvtz', software='gaussian'),
                                                     energy=arkane.modelchem.LevelOfTheory(method='dlpno-ccsd(t)-f12-2023', basis='cc-pvtz-f12', software='orca')),
    'M062X/ccpvtz': arkane.modelchem.LevelOfTheory(method='M062X',
                           basis='ccpvtz',
                           software='gaussian',
                           ),
    'dlpno-ccsd(t)-f12-2023': arkane.modelchem.LevelOfTheory(method='dlpnoccsd(t)f122023',
                           basis='ccpvtzf12',
                           software='orca',
                           ),
}

freq_scale_factors = {
    'M062X/ccpvtz': 0.955,
    'M062X/def2tzvp': 0.984,
    'dlpno-ccsdt-f12-ccpvtz': 1.002,
}
def log_to_enthalpy(method, freq_lot, energy_lot, freq_log, energy_log=None, energy_dict=None, temp=298.15):

    freq_log = arkane.ess.ess_factory(freq_log)
    
    conformer, _ = freq_log.load_conformer()
    
    # Perform quick checks
    assert conformer.spin_multiplicity > 0
    assert any(isinstance(mode, rmgpy.statmech.IdealGasTranslation) for mode in conformer.modes)
    assert any(isinstance(mode, (rmgpy.statmech.LinearRotor, rmgpy.statmech.NonlinearRotor)) for mode in conformer.modes)
    assert any(isinstance(mode, rmgpy.statmech.HarmonicOscillator) for mode in conformer.modes)
    
    coords, nums, masses = freq_log.load_geometry()
    assert len(nums) > 1
    
    atoms = collections.Counter([arkane.common.symbol_by_number[int(n)] for n in nums])
    conformer.coordinates = (coords, 'angstroms')
    conformer.number = nums
    conformer.mass = (masses, 'amu')
    
    freq_scale_factor = freq_scale_factors[method]
    frequencies = conformer.modes[2].frequencies.value_si
    for mode in conformer.modes:
        if isinstance(mode, rmgpy.statmech.HarmonicOscillator):
            mode.frequencies = (frequencies * freq_scale_factor, "cm^-1")
    if freq_scale_factor == 1:
        print('WARNING: Frequency scale factor is 1')
    zpe_scale_factor = freq_scale_factor / 1.014
    
    # get electronic energy (choose one option)
    # option 1: read from the log
    if energy_log is not None:
        energy_log = arkane.ess.ess_factory(energy_log)
        energy = energy_log.load_energy(zpe_scale_factor=zpe_scale_factor)  # J/mol

    # option 2: just read from a dictionary provided 
#     energy = energy_dict[int(label)] * 627.5094740631 * 4184
    
    # add ZPE
    energy += freq_log.load_zero_point_energy() * zpe_scale_factor if len(nums) > 1 else 0  # J/mol
    
    # add AECs
#     print(get_atom_correction(energy_lot, atoms) )
    energy += arkane.encorr.corr.get_atom_correction(energy_lot, atoms)  # J/mol
    conformer.E0 = (energy / 4184, 'kcal/mol')
    
    return rmgpy.quantity.ScalarQuantity((conformer.get_enthalpy(temp) + conformer.E0.value_si) / 4184, 'kcal/mol')



# freq_log = '/scratch/harris.se/guassian_scratch/bac/species_0010/sp.log'
# energy_log = '/scratch/harris.se/guassian_scratch/bac/species_0010/conformer.out'
# log_to_enthalpy(method, freq_lot, energy_lot, freq_log, energy_log=energy_log, energy_dict=None, temp=298.15)


## Sanity Check on the calculations I've run

In [ ]:
# make sure the orca geometry has the right atom counts

for i in complete_sp:
    orca_log = os.path.join(working_dir, f'species_{i:04}', 'conformer.out')
    geo, nums, mass = arkane.ess.ess_factory(orca_log).load_geometry()
    atoms = collections.Counter([arkane.common.symbol_by_number[int(n)] for n in nums])
    
    ref_sp = rmgpy.species.Species().from_adjacency_list(database.reference_sets['main'][i].adjacency_list)
    ref_nums = [atom.number for atom in ref_sp.molecule[0].atoms]
    atoms2 = collections.Counter([arkane.common.symbol_by_number[int(n)] for n in ref_nums])
    if atoms != atoms2:
        print(i, 'difference!')

In [ ]:
# look at any species where the H298 values vary by more than 5kcal/mol
for i in complete_sp:
#     if i ==219:
#         continue
    
    
    energy_log = os.path.join(working_dir, f'species_{i:04}', 'conformer.out')
    freq_log = os.path.join(working_dir, f'species_{i:04}', 'freq.log')
    if not (os.path.exists(energy_log) and os.path.exists(freq_log)):
        print(f'missing {i} files')
        continue
    
    ref_sp = rmgpy.species.Species().from_adjacency_list(database.reference_sets['main'][i].adjacency_list)
    
    
    method = 'M062X/ccpvtz'
    freq_lot = arkane.modelchem.LevelOfTheory(method='M062X', basis='ccpvtz', software='gaussian')
    energy_lot = arkane.modelchem.LevelOfTheory(method='dlpno-ccsd(t)-f12-2023', basis='ccpvtzf12', software='orca')

    hf298 = log_to_enthalpy(method, freq_lot, energy_lot, freq_log, energy_log=energy_log, temp=298.15)    
    
    
    # see if ref_db is off by too much
    my_key = [key for key in database.reference_sets['main'][i].reference_data.keys()][0]
    ref_h298 = database.reference_sets['main'][i].reference_data[my_key].thermo_data.H298
    kcal_diff = np.abs(hf298.value_si - ref_h298.value_si) / 4184
    if kcal_diff > 3.0:
        print(i, f'{kcal_diff:.3f} kcal\t\t{hf298.value_si / 4184:.3f}\t{ref_h298.value_si / 4184:.3f}')

# Prepare the database for new additions

In [ ]:
ref_spcs = {spc.index: spc for spc in database.reference_sets['main']}

In [ ]:
lot = arkane.modelchem.CompositeLevelOfTheory(
    freq=arkane.modelchem.LevelOfTheory(method='m062x',basis='ccpvtz',software='gaussian'),
    energy=arkane.modelchem.LevelOfTheory(method='dlpno-ccsd(t)-f12-2023',basis='ccpvtzf12',software='orca')
)


In [ ]:
database.reference_sets['main'][i].reference_data['ATcT'].thermo_data.H298.value_si

## Clean out previous attempts from the DB

In [ ]:
already_in_db = []
for i in range(len(database.reference_sets['main'])):

    if lot in database.reference_sets['main'][i].calculated_data.keys():
        already_in_db.append(i)
        
print(f'Found out {len(already_in_db)} entries')
    

In [ ]:
clearing_out = []
for i in range(len(database.reference_sets['main'])):

    if lot in database.reference_sets['main'][i].calculated_data.keys():
        clearing_out.append(i)
        database.reference_sets['main'][i].calculated_data.pop(lot)
        
print(f'Cleared out {len(clearing_out)} entries')
    

In [ ]:
skip = []
added_to_database = []
for i in complete_sp:
    if i in skip:
        continue
    
    energy_log = os.path.join(working_dir, f'species_{i:04}', 'conformer.out')
    freq_log = os.path.join(working_dir, f'species_{i:04}', 'freq.log')
    if not (os.path.exists(energy_log) and os.path.exists(freq_log)):
        print(f'missing {i} files')
        continue
    
    ref_sp = rmgpy.species.Species().from_adjacency_list(database.reference_sets['main'][i].adjacency_list)
    method = 'M062X/ccpvtz'
    freq_lot = arkane.modelchem.LevelOfTheory(method='M062X', basis='ccpvtz', software='gaussian')
    energy_lot = arkane.modelchem.LevelOfTheory(method='dlpno-ccsd(t)-f12-2023', basis='ccpvtzf12', software='orca')
    hf298 = log_to_enthalpy(method, freq_lot, energy_lot, freq_log, energy_log=energy_log, temp=298.15)    
    
    
    # see if ref_db is off by too much
    my_key = [key for key in database.reference_sets['main'][i].reference_data.keys()][0]
    ref_h298 = database.reference_sets['main'][i].reference_data[my_key].thermo_data.H298
    kcal_diff = np.abs(hf298.value_si - ref_h298.value_si) / 4184
    if kcal_diff > 6.0:
        print(i, f'{kcal_diff:.3f} kcal\t\t{hf298.value_si / 4184:.3f}\t{ref_h298.value_si / 4184:.3f}')
    

    # Update thermo
    try:
        database.reference_sets['main'][i].calculated_data[lot] = arkane.encorr.reference.CalculatedDataEntry(
            rmgpy.thermo.ThermoData(H298=hf298),
            xyz_dict=log_to_xyz_dict(energy_log)
        )
    except KeyError:
        print(f'Failed to add {i} to the database')
        continue
    added_to_database.append(i)
#     print(f'Added {i} to the database')
print(f'Added {len(added_to_database)} entries to database')

In [ ]:
database.save()

In [ ]:
my_key = [key for key in database.reference_sets['main'][i].reference_data.keys()][0]

In [ ]:
n = 100
database.reference_sets['main'][n].smiles

In [ ]:
database.reference_sets['main'][n].reference_data[my_key].thermo_data.H298.value_si / 4184

In [ ]:
keys = list(database.reference_sets['main'][n].calculated_data.keys())

In [ ]:
ref_sp = rmgpy.species.Species(smiles=database.reference_sets['main'][n].smiles)

In [ ]:
database_fun.get_unique_species_index(ref_sp)

In [ ]:
hf298

In [ ]:
for key in database.reference_sets['main'][n].calculated_data.keys():
    print(database.reference_sets['main'][n].calculated_data[key].thermo_data.H298)

In [ ]:
key = keys[3]
#     print(database.reference_sets['main'][n].calculated_data[key].xyz_dict)

coords = database.reference_sets['main'][n].calculated_data[key].xyz_dict['coords']
syms = database.reference_sets['main'][n].calculated_data[key].xyz_dict['symbols']
nums = [periodic_table.GetAtomicNumber(sym) for sym in syms]
atoms = ase.Atoms(nums, coords)


ase.visualize.view(atoms, viewer='x3d')


In [ ]:
print(autotst_wrapper.get_xyz(atoms))

In [ ]:
lot

# Actually run BAC Fitting

In [ ]:
my_bac_job = arkane.encorr.bac.BACJob(
    lot,  # level of theory
    exclude_elements=['S', 'N', 'Cl', 'F']
)
my_bac_job.execute()

In [ ]:
my_bac_job.bac.fit(exclude_elements=['S', 'N', 'Cl', 'F'])

In [ ]:
my_bac_job.bac.bacs

In [ ]:
my_bac_job.bac.confidence_intervals

In [ ]:
# my_bac_job.bac.fit(exclude_elements=['S', 'N', 'Cl', 'F'])

In [ ]:
my_bac_job.write_output('/projects/westgroup/harris.se/autoscience/reaction_calculator/util/new')

In [ ]:
for a in :
    print(a.mol)

In [ ]:
my_bac_job.bac.dataset[2].mol

In [ ]:
my_bac_job.bac.dataset[2].calc_data

In [ ]:
my_bac_job.bac.dataset[2].bac_data

In [ ]:
dir(my_bac_job.bac.dataset[2])

In [ ]:
my_bac_job.bac.dataset[2].level_of_theory

In [2]:
df = pd.read_csv('new/corrections.csv')

In [3]:
original_enthalpies = df['Calculated Enthalpy'].values
corrected_enthalpies = df['Corrected Enthalpy'].values
ref_enthalpies = df['Reference Enthalpy'].values
smiles = df['Smiles'].values

In [ ]:
for i in range(len(df)):
    difference = corrected_enthalpies[i] - original_enthalpies[i]
    if smiles[i] == 'CCCC':
        print(i)
    # if np.abs(difference) > 0.1:
    #     print(f'{difference:.2f}\t{smiles[i]}')

In [ ]:
# how many of the species the reference database are calculated with rotors?
add_to = []
for i in range(len(smiles)):

    sp = rmgpy.species.Species(smiles=smiles[i])
    try:
        print(database_fun.get_unique_species_index(sp))
    except IndexError:
        add_to.append(sp)




In [ ]:
database_fun.add_species_to_database(add_to)

In [21]:
calced = []
torsions = 0
for i in range(len(smiles)):

    sp = rmgpy.species.Species(smiles=smiles[i])
    species_index = database_fun.get_unique_species_index(sp)
    thermolib = os.path.join(os.environ['DFT_DIR'], 'thermo', f'species_{species_index:04}', 'arkane', 'RMG_libraries', 'thermo.py')
    if os.path.exists(thermolib):
        # print(species_index)
        calced.append(species_index)
    else:
        sp1 = autotst.species.Conformer(smiles=smiles[i])
        torsions += len(sp1.get_torsions())
    


In [22]:
torsions

178

In [6]:
len(calced) / len(smiles)

0.20915032679738563

In [9]:
len(calced)

32

In [10]:
len(smiles)

153

In [14]:
import autotst.species

In [17]:
sp1 = autotst.species.Conformer(smiles=smiles[0])

In [18]:
sp1.get_torsions()

[<Torsion "(14, 0, 1, 2)">,
 <Torsion "(0, 1, 2, 4)">,
 <Torsion "(0, 1, 3, 11)">,
 <Torsion "(1, 2, 4, 8)">]

In [ ]:
# Get the lowest energy conformer from the overall result -- look in the arkane folder
conformer_file = autotst_wrapper.get_lowest_valid_conformer(conformer_dir, species_index)

new_cf = autotst.species.Conformer(smiles=smiles)  # TODO make this from adjacency list?
new_cf._ase_molecule = autotst_wrapper.get_gaussian_file_geometry(conformer_file)
new_cf.update_coords_from(mol_type="ase")
torsions = new_cf.get_torsions()  # TODO - is this only the nonterminal ones?


In [11]:
smiles

array(['CCC(C)O', 'CC=CCC', 'C=CCC', 'C#CCC', 'C[CH]O', 'CC1=CCCC1',
       'C=[C]C', 'CCCCCO', 'CCCO', '[CH]=CC', '[C]#CC', 'COC(C)OC',
       'C=C=CC', 'COCCOC', 'C=CC=C', 'C#CC#C', 'C1OCOCO1', 'C1COCCO1',
       'C=CCCC=C', 'CC#CC', '[CH2]CO', 'COC(C)C', 'CC(C)CO', 'CC(C)O',
       'C#C[CH2]', '[C]C#C', 'CC(=O)C(C)=O', 'CC(C)C(C)C', 'CC(=O)C(C)C',
       'C=C(CC)CC', 'C1=COCCC1', 'C=C1CC(=O)O1', 'CC=O', 'CC(=O)O',
       'CC(C)=O', '[CH2]C(C)=O', 'C[C]=O', 'C#C', 'C1C2CC3CC1CC(C2)C3',
       'C=C=C', '[CH2]C=C', 'COC1=CC=CC=C1', 'O=CC1=CC=CC=C1',
       'C1=CC=CC=C1', '[CH2]C1=CC=CC=C1', 'C1C2CC12', 'O=C=O',
       '[C-]#[O+]', 'O=C(O)O', 'C1CCC1', 'C1=CCC1', 'C1CCCCC1',
       'O=C1CCCCC1', 'C1=CCC=C1', 'C1CCCC1', 'C1=CCCC1', '[C]1=CC1',
       '[CH]1C=C1', 'C1CC1', 'C1=CC1', 'C1=CC=1', '[H][H]', 'COCOC',
       'COC', '[O]O', 'C1OO1', '[O][O]', 'C=[O+][O-]', 'CC', 'CCO',
       'C=CO', 'CC[O]', 'C=COCC', 'CCOC(C)=O', 'C[CH2]', 'CCC1=CC=CC=C1',
       'CCC1CCC1', 'OCCO', 'C=C', '[C

In [ ]:
pd.DataFrame({'i': [1], 'nononon': 2})

In [ ]:
print(rmgpy.species.Species(smiles=smiles[3]).to_adjacency_list())

In [ ]:
print(rmgpy.species.Species(smiles=smiles[2]).to_adjacency_list())

In [ ]:
smiles[2]

In [ ]:
corrected_enthalpies[2]

In [ ]:
original_enthalpies[2]

In [ ]:
corrected_enthalpies[141]

In [ ]:
original_enthalpies[141]

In [ ]:
ref_enthalpies[141]

In [ ]:
corrected_enthalpies[141] - original_enthalpies[141]

In [ ]:
df.columns